# Gerador de Instâncias

In [ ]:
import pandas as pd
import random
import math
from haversine import haversine, Unit

class Instancia:
    def __init__(self, cenario_estoque, seed=42):
        random.seed(seed)
        self.cenario = cenario_estoque
        self.num_clientes = 500  # O artigo pede 500 clientes
        self.num_produtos_total = 100 # O artigo pede 100 produtos

        # Estruturas de dados
        self.produtos = []
        self.pesos = {}
        self.clientes = []
        self.locais = {}
        self.distancias = {}
        self.pedidos = {}
        self.estoque = {}
        self.veiculos = {'pequeno': 0, 'medio': 0, 'grande': 4}

        self.CDs = ['CD1', 'CD2']
        self.custos_veiculos = {'pequeno': 1, 'medio': 2, 'grande': 3}
        self.cap_veiculos = {'pequeno': 10, 'medio': 1200, 'grande': 7500}

    def carregar_dados(self, caminho_csv_enderecos, caminho_csv_produtos):
        """
        Lê dados reais dos CSVs.
        Ajustado para colunas: name, latitude, longitude
        """
        print(f"--- Carregando Dados Reais ---")

        # --- 1. CARREGAR PRODUTOS (Olist) ---
        # (Essa parte continua igual, pois o arquivo Olist não mudou)
        print(f"Lendo produtos de: {caminho_csv_produtos}")
        df_prod = pd.read_csv(caminho_csv_produtos)
        df_prod = df_prod[df_prod['product_weight_g'] > 0].dropna(subset=['product_weight_g'])

        if len(df_prod) < self.num_produtos_total:
            raise ValueError(f"O arquivo de produtos tem apenas {len(df_prod)} itens válidos.")

        produtos_selecionados = df_prod.sample(n=self.num_produtos_total, random_state=42)

        for _, row in produtos_selecionados.iterrows():
            p_id = row['product_id']
            peso_kg = row['product_weight_g'] / 1000.0
            peso_kg = max(peso_kg, 0.1)
            self.produtos.append(p_id)
            self.pesos[p_id] = peso_kg

        # --- 2. CARREGAR ENDEREÇOS (Seus arquivos Instance-Coordinates) ---
        print(f"Lendo endereços de: {caminho_csv_enderecos}")
        df_end = pd.read_csv(caminho_csv_enderecos)

        # AJUSTE AQUI: Usando 'latitude' e 'longitude' (minúsculo)
        df_end = df_end.dropna(subset=['latitude', 'longitude'])

        # Precisamos de 500 clientes + 2 CDs = 502 pontos
        total_pontos = self.num_clientes + 2

        if len(df_end) < total_pontos:
            raise ValueError(f"Arquivo tem apenas {len(df_end)} pontos. Precisa de {total_pontos}.")

        # Selecionar pontos aleatórios
        pontos_selecionados = df_end.sample(n=total_pontos, random_state=42)
        lista_pontos = pontos_selecionados.to_dict('records')

        # =================================================================
        # 3. DEFINIR LOCALIZAÇÕES DOS CDs
        # =================================================================
        cd1_ponto = lista_pontos[0]
        cd2_ponto = lista_pontos[1]

        # Fixa CDs em no centro da localização dos clientes
        self.locais['CD1'] = (-23.42827702, -46.43182168)
        self.locais['CD2'] = (-23.44458995, -46.53164056)

        # Define os restantes 500 como Clientes
        for i in range(self.num_clientes):
            dados = lista_pontos[i]
            c_id = f"cli_{i}"
            self.clientes.append(c_id)
            # AJUSTE AQUI TAMBÉM
            self.locais[c_id] = (dados['latitude'], dados['longitude'])

        print(f"Sucesso! {len(self.produtos)} produtos e {len(self.clientes)} clientes carregados.")

        # --- 3. CALCULAR DISTÂNCIAS ---
        print("Calculando matriz de distâncias...")
        pontos = self.CDs + self.clientes
        for i in pontos:
            for j in pontos:
                if i != j:
                    dist = haversine(self.locais[i], self.locais[j], unit=Unit.KILOMETERS)
                    self.distancias[(i,j)] = dist

    def gerar_pedidos(self):
        """
        Gera os pedidos seguindo a distribuição da Tabela 1.
        """
        print("--- Gerando Pedidos (Tabela 1) ---")
        demandas_por_produto = {p: 0 for p in self.produtos}
        peso_total_pedidos = 0

        # Classes da Tabela 1: (Faixa de qtd, Probabilidade)
        opcoes = ['c1', 'c2', 'c3', 'c4']
        pesos_prob = [0.50, 0.25, 0.08, 0.16]

        for cli in self.clientes:
            classe = random.choices(opcoes, weights=pesos_prob, k=1)[0]

            if classe == 'c1': qtd_itens = random.randint(1, 2)
            elif classe == 'c2': qtd_itens = random.randint(2, 3)
            elif classe == 'c3': qtd_itens = random.randint(3, 4)
            else: qtd_itens = random.randint(4, 10)

            # Escolher aleatoriamente QUAIS produtos
            prods_escolhidos = random.sample(self.produtos, k=qtd_itens)

            self.pedidos[cli] = {}
            for p in prods_escolhidos:
                # Artigo não especifica qtd do MESMO produto, assumiremos 1 unidade
                qtd = 1
                self.pedidos[cli][p] = qtd
                demandas_por_produto[p] += qtd
                peso_total_pedidos += self.pesos[p] * qtd

        self.peso_total_pedidos = peso_total_pedidos
        self.demandas_totais = demandas_por_produto
        print("Pedidos gerados.")

    def gerar_estoque(self):
        """
        Gera estoques seguindo a Tabela 2 e Seção 4.1.
        """
        print(f"--- Gerando Estoque (Cenário {self.cenario}) ---")

        self.estoque['CD1'] = {p: 0 for p in self.produtos}
        self.estoque['CD2'] = {p: 0 for p in self.produtos}

        for p in self.produtos:
            demanda = self.demandas_totais[p]
            if demanda == 0: continue

            # Ocorrência: 33% CD1, 33% CD2, 33% Ambos
            onde = random.choice(['CD1', 'CD2', 'AMBOS'])

            if onde == 'CD1':
                fator = 1.10 if self.cenario in [1, 2] else 1.05
                self.estoque['CD1'][p] = math.ceil(demanda * fator)

            elif onde == 'CD2':
                fator = 1.10 if self.cenario in [1, 2] else 1.05
                self.estoque['CD2'][p] = math.ceil(demanda * fator)

            else: # AMBOS
                if self.cenario == 1: # 70% cada
                    qtd = math.ceil(demanda * 0.70)
                    self.estoque['CD1'][p] = qtd
                    self.estoque['CD2'][p] = qtd
                elif self.cenario == 2: # 80% cada
                    qtd = math.ceil(demanda * 0.80)
                    self.estoque['CD1'][p] = qtd
                    self.estoque['CD2'][p] = qtd
                elif self.cenario == 3: # VA e 100-VA
                    va = random.randint(10, 100) # Valor entre 10 e 100
                    perc_cd1 = va / 100.0
                    perc_cd2 = (100 - va) / 100.0
                    self.estoque['CD1'][p] = math.ceil(demanda * perc_cd1)
                    # Nota: O artigo diz "100% - VA", implicando soma da demanda total
                    self.estoque['CD2'][p] = math.ceil(demanda * perc_cd2)

    def calcular_frota(self):
        """
        Calcula qtd de veículos conforme Seção 4.1.
        """
        # Pequenos: dobro do número de pedidos
        qtd_pedidos = len(self.pedidos)
        self.veiculos['pequeno'] = qtd_pedidos * 2

        # Médios: teto(1.5 * PesoTotal / 1200)
        sp = self.peso_total_pedidos
        self.veiculos['medio'] = math.ceil(1.5 * (sp / 1200))

        # Grandes: fixo em 4
        self.veiculos['grande'] = 4

        print(f"Frota Calculada: Peq={self.veiculos['pequeno']}, Med={self.veiculos['medio']}, Gra={self.veiculos['grande']}")

# --- EXECUÇÃO DE TESTE ---
if __name__ == "__main__":
    # Exemplo de teste com arquivos reais
    try:
        inst = Instancia(cenario_estoque=1)

        ARQUIVO_ENDERECOS = "/content/Instance-Coordinates-0.csv"
        ARQUIVO_PRODUTOS = "/content/olist_products_dataset.csv"

        inst.carregar_dados(ARQUIVO_ENDERECOS, ARQUIVO_PRODUTOS)

        inst.gerar_pedidos()
        inst.gerar_estoque()
        inst.calcular_frota()

        print("\n--- Verificação de Dados ---")
        # Pega um cliente qualquer para testar
        cliente_teste = inst.clientes[0]
        print(f"Exemplo Pedido Cliente {cliente_teste}: {inst.pedidos[cliente_teste]}")

        prod_exemplo = list(inst.pedidos[cliente_teste].keys())[0]
        print(f"Estoque Produto {prod_exemplo}: CD1={inst.estoque['CD1'][prod_exemplo]}, CD2={inst.estoque['CD2'][prod_exemplo]}")
        print("Teste do gerador concluído com sucesso!")

    except FileNotFoundError:
        print(f"ERRO: Não encontrei os arquivos.")
        print(f"Verifique se existe a pasta 'data' e se os arquivos estão dentro dela:")
        print(f" - data/Instance-Coordinates-0.csv")
        print(f" - data/olist_products_dataset.csv")
    except Exception as e:
        print(f"Erro no teste: {e}")

--- Carregando Dados Reais ---
Lendo produtos de: /content/olist_products_dataset.csv
Lendo endereços de: /content/Instance-Coordinates-0.csv
Sucesso! 100 produtos e 500 clientes carregados.
Calculando matriz de distâncias...
--- Gerando Pedidos (Tabela 1) ---
Pedidos gerados.
--- Gerando Estoque (Cenário 1) ---
Frota Calculada: Peq=1000, Med=4, Gra=4

--- Verificação de Dados ---
Exemplo Pedido Cliente cli_0: {'2ba5ea51230217603c8a13fc5dc24e88': 1, '758c721416d9d8799cb3059bd648240d': 1}
Estoque Produto 2ba5ea51230217603c8a13fc5dc24e88: CD1=9, CD2=9
Teste do gerador concluído com sucesso!


# ALNS

In [ ]:
import copy
import math
import random

class Entrega:
    """
    Representa uma entrega única para um cliente.
    """
    def __init__(self, id_cliente, cd_origem, veiculo, produtos_entregues, dist_km):
        self.id_cliente = id_cliente
        self.cd_origem = cd_origem
        self.veiculo = veiculo  # 'pequeno', 'medio', 'grande'
        self.produtos = produtos_entregues # Dicionário {prod: qtd}
        self.dist_km = dist_km
        self.custo = 0.0

    def calcular_custo_entrega(self, custos_veiculo):
        # Custo = (Custo/km do veículo * Distância)
        # O artigo não cita custo fixo explicitamente na fórmula (1), apenas c_ij^k
        # Assumiremos c_ij^k como (taxa_veiculo * distancia)
        taxa = custos_veiculo[self.veiculo]
        self.custo = taxa * self.dist_km
        return self.custo

class Solucao:
    """
    Representa uma solução completa (conjunto de entregas).
    """
    def __init__(self, instancia):
        self.instancia = instancia
        self.entregas = [] # Lista de objetos Entrega
        self.custo_total = 0.0
        self.entregas_por_cliente = {c: [] for c in instancia.clientes}
        self.veiculos_utilizados = {
            "pequeno": 0,
            "medio": 0,
            "grande": 0
        }


        # Controle de Estoque Atual (copia do original para ir decrementando)
        self.estoque_atual = copy.deepcopy(instancia.estoque)

    def adicionar_entrega(self, entrega):
        # Atualiza custo
        custo = entrega.calcular_custo_entrega(self.instancia.custos_veiculos)
        self.custo_total += custo
        self.entregas.append(entrega)
        self.entregas_por_cliente[entrega.id_cliente].append(entrega)
        self.veiculos_utilizados[entrega.veiculo] += 1

        # Atualiza Estoque Virtual da Solução
        for prod, qtd in entrega.produtos.items():
            self.estoque_atual[entrega.cd_origem][prod] -= qtd

    def validar_viabilidade(self):
        # Verifica se não negativou estoque
        for cd in self.estoque_atual:
            for prod, qtd in self.estoque_atual[cd].items():
                if qtd < 0: return False
        return True

    def recalcular_frota_do_zero(self):
        """
        Zera e recontabiliza os veículos baseados na lista real de entregas.
        Serve para corrigir erros de acumulação na heurística.
        """
        self.veiculos_utilizados = {"pequeno": 0, "medio": 0, "grande": 0}
        self.custo_total = 0.0

        for entrega in self.entregas:
            self.veiculos_utilizados[entrega.veiculo] += 1
            self.custo_total += entrega.custo

def gerar_solucao_gulosa(instancia):
    """
    Cria uma solução inicial tentando sempre o CD mais próximo.
    Se não tiver estoque, tenta o outro.
    (Versão simplificada para Modelo 1: Entrega Direta)
    """
    solucao = Solucao(instancia)
    print("--- Gerando Solução Inicial (Gulosa) ---")

    # Itera sobre todos os clientes e seus pedidos
    for cliente, pedido in instancia.pedidos.items():

        # 1. Descobrir qual veículo usar (baseado no peso total do pedido)
        peso_pedido = sum(pedido[p] * instancia.pesos[p] for p in pedido)
        tipo_veiculo = 'pequeno' if peso_pedido <= 10 else 'medio'

        # 2. Definir qual CD é mais perto
        dist_cd1 = instancia.distancias[('CD1', cliente)]
        dist_cd2 = instancia.distancias[('CD2', cliente)]

        # Ordena CDs por proximidade: [(CD_Nome, Distancia), (Outro_CD, Distancia)]
        cds_ordenados = sorted([('CD1', dist_cd1), ('CD2', dist_cd2)], key=lambda x: x[1])

        atendido = False

        # Tenta atender pelo CD mais próximo primeiro
        for cd_nome, dist in cds_ordenados:
            # Verifica se esse CD tem estoque para TODO o pedido
            tem_estoque = True
            for prod, qtd in pedido.items():
                if solucao.estoque_atual[cd_nome][prod] < qtd:
                    tem_estoque = False
                    break

            if tem_estoque:
                # Cria a entrega
                nova_entrega = Entrega(cliente, cd_nome, tipo_veiculo, pedido, dist)
                solucao.adicionar_entrega(nova_entrega)
                atendido = True
                break # Sai do loop dos CDs, vai para próximo cliente

        if not atendido:
            # CASO CRÍTICO: Nenhum CD tem o pedido completo sozinho.
            # Como é o Modelo 1, podemos fazer Split Delivery (entregar o que der de um, resto do outro)
            # Lógica Simples: Pega tudo que der do mais próximo, resto do outro.

            cd_pref, dist_pref = cds_ordenados[0]
            cd_seg, dist_seg = cds_ordenados[1]

            pedido_restante = pedido.copy()
            prods_cd1 = {}

            # Enche o carrinho no CD preferido
            for prod, qtd in pedido.items():
                disponivel = solucao.estoque_atual[cd_pref][prod]
                if disponivel > 0:
                    qtd_pegar = min(qtd, disponivel)
                    prods_cd1[prod] = qtd_pegar
                    pedido_restante[prod] -= qtd_pegar
                    if pedido_restante[prod] == 0:
                        del pedido_restante[prod]

            # Registra entrega
            if prods_cd1:
                peso = sum(prods_cd1[p] * instancia.pesos[p] for p in prods_cd1)
                veic = 'pequeno' if peso <= 10 else 'medio'
                solucao.adicionar_entrega(Entrega(cliente, cd_pref, veic, prods_cd1, dist_pref))

            # O que sobrou, pega do CD secundário
            if pedido_restante:
                peso = sum(pedido_restante[p] * instancia.pesos[p] for p in pedido_restante)
                veic = 'pequeno' if peso <= 10 else 'medio'
                solucao.adicionar_entrega(Entrega(cliente, cd_seg, veic, pedido_restante, dist_seg))

    return solucao

# --- OPERADORES DE DESTRUIÇÃO (REMOVE) ---
def destroy_random(solucao_original, num_remover):
    """
    Remove 'num_remover' entregas aleatórias da solução.
    Retorna: (solucao_parcial, lista_clientes_removidos)
    """
    solucao = copy.deepcopy(solucao_original)

    # Se pedir para remover mais do que tem, ajusta
    qtd = min(num_remover, len(solucao.entregas))

    # Escolhe índices aleatórios para remover
    indices_para_remover = random.sample(range(len(solucao.entregas)), qtd)

    entregas_removidas = []

    # Ordenamos índices decrescentes para remover sem quebrar a lista
    for idx in sorted(indices_para_remover, reverse=True):
        entrega = solucao.entregas.pop(idx)

        # 1. Devolve o estoque para o sistema
        for prod, q_prod in entrega.produtos.items():
            # Nota: Se foi transbordo, a gente descontou do CD de "estoque lógico"
            # Precisamos saber de onde descontou.
            # Na sua classe Entrega, seria ideal guardar 'cd_estoque_real' além do 'cd_origem_fisica'.
            # Mas assumindo que no seu repair você descontou do CD correto, aqui devolvemos:

            # Simplificação: Como no repair você usou cd_desconta_estoque,
            # precisamos garantir que aqui estamos devolvendo para o lugar certo.
            # Se não tiver essa info na Entrega, use o cd_origem.
            solucao.estoque_atual[entrega.cd_origem][prod] += q_prod

        # 2. Remove custo
        solucao.custo_total -= entrega.custo

        # --- CORREÇÃO AQUI: Remove contagem do veículo ---
        if entrega.veiculo in solucao.veiculos_utilizados:
            solucao.veiculos_utilizados[entrega.veiculo] -= 1

        entregas_removidas.append(entrega)

    clientes_removidos = [ent.id_cliente for ent in entregas_removidas]

    return solucao, clientes_removidos

def repair_greedy(solucao, clientes_removidos, permitir_fracionamento=True, permitir_troca=True):
    instancia = solucao.instancia

    # Limites e Custos
    dist_entre_cds = instancia.distancias[('CD1', 'CD2')]
    custo_km_grande = instancia.custos_veiculos['grande']
    cap_grande = instancia.cap_veiculos['grande']
    cap_medio = instancia.cap_veiculos['medio'] # 1200kg
    cap_pequeno = instancia.cap_veiculos['pequeno'] # 10kg

    # Limites da Frota (Hard Constraint)
    limite_medios = instancia.veiculos['medio']

    # 1. ORDENAÇÃO ESTRATÉGICA:
    # Processa os pedidos mais pesados primeiro para garantir veículos médios onde são obrigatórios.
    clientes_ordenados = sorted(clientes_removidos,
                                key=lambda c: sum(instancia.pedidos[c][p] * instancia.pesos[p] for p in instancia.pedidos[c]),
                                reverse=True)

    for cliente in clientes_ordenados:
        pedido = instancia.pedidos[cliente]
        melhor_opcao = None
        menor_custo_total = float('inf')

        peso_pedido = sum(pedido[p] * instancia.pesos[p] for p in pedido)

        # Define tipo de veículo necessário
        tipo_ideal = 'pequeno' if peso_pedido <= cap_pequeno else 'medio'

        # VERIFICAÇÃO DE DISPONIBILIDADE DA FROTA
        veiculo_viavel = tipo_ideal
        if tipo_ideal == 'medio':
            qtd_em_uso = solucao.veiculos_utilizados['medio']
            if qtd_em_uso >= limite_medios:
                # Não há veículos médios disponíveis.
                # Tentativa de Fallback: O pedido cabe em 2 Motos (Split)?
                # Capacidade total de 2 motos = 20kg.
                if permitir_fracionamento and peso_pedido <= (2 * cap_pequeno):
                    veiculo_viavel = 'SPLIT_FORCADO' # Força a lógica de split lá embaixo
                else:
                    # Se for > 20kg e não tem caminhão médio, cliente fica sem atendimento (inviável)
                    # Na ALNS, isso geralmente é penalizado ou deixado na lista de não atendidos.
                    continue

        taxa_entrega = instancia.custos_veiculos[tipo_ideal if veiculo_viavel != 'SPLIT_FORCADO' else 'pequeno']

        # --- TENTATIVA 1: ENTREGA ÚNICA (Direta ou Transbordo) ---
        # Só executa se não estamos forçando split por falta de caminhão
        if veiculo_viavel != 'SPLIT_FORCADO':
            for cd_saida_estoque in ['CD1', 'CD2']:
                cd_transbordo = 'CD2' if cd_saida_estoque == 'CD1' else 'CD1'

                # Verifica Estoque
                tem_estoque = True
                for prod, qtd in pedido.items():
                    if solucao.estoque_atual[cd_saida_estoque][prod] < qtd:
                        tem_estoque = False; break

                if not tem_estoque: continue

                # A. Direta
                dist_direta = instancia.distancias[(cd_saida_estoque, cliente)]
                custo_direto = taxa_entrega * dist_direta

                if custo_direto < menor_custo_total:
                    menor_custo_total = custo_direto
                    melhor_opcao = (cd_saida_estoque, veiculo_viavel, pedido, dist_direta, False)

                # B. Transbordo
                if permitir_troca:
                    dist_vizinho_cliente = instancia.distancias[(cd_transbordo, cliente)]
                    custo_perna_final = taxa_entrega * dist_vizinho_cliente

                    fator_ocupacao = peso_pedido / cap_grande
                    custo_transferencia = max(
                        0.10 * (custo_km_grande * dist_entre_cds),
                        fator_ocupacao * (custo_km_grande * dist_entre_cds)
                    )
                    custo_com_troca = custo_transferencia + custo_perna_final

                    if custo_com_troca < menor_custo_total:
                        menor_custo_total = custo_com_troca
                        melhor_opcao = (cd_transbordo, veiculo_viavel, pedido, dist_vizinho_cliente, True, cd_saida_estoque)

        # --- APLICA ENTREGA ÚNICA (Se encontrada) ---
        if melhor_opcao:
            if len(melhor_opcao) == 5:
                cd, veic, ped, d, _ = melhor_opcao
                solucao.adicionar_entrega(Entrega(cliente, cd, veic, ped, d))
            else:
                cd_fisico, veic, ped, d, _, cd_estoque = melhor_opcao
                entrega = Entrega(cliente, cd_fisico, veic, ped, d)

                peso = sum(ped[p] * instancia.pesos[p] for p in ped)
                fator = peso / cap_grande
                custo_transf = max(0.10 * (custo_km_grande * dist_entre_cds), fator * (custo_km_grande * dist_entre_cds))
                entrega.custo += custo_transf

                solucao.custo_total += entrega.custo
                solucao.entregas.append(entrega)
                solucao.entregas_por_cliente[entrega.id_cliente].append(entrega)
                solucao.veiculos_utilizados[veic] += 1

                for prod, qtd in ped.items():
                    solucao.estoque_atual[cd_estoque][prod] -= qtd

        # --- TENTATIVA 2: SPLIT DELIVERY (Fallback ou Lógica de Estoque) ---
        elif permitir_fracionamento:
            # Lógica: Tenta pegar o máximo do CD mais próximo (ou CD1), resto do outro
            # Mas agora usando VEÍCULOS PEQUENOS obrigatoriamente se faltou caminhão médio

            veiculo_uso = 'pequeno' # No split geralmente usamos veículos menores para completar

            # Se for > 10kg, o split VAI exigir 2 veículos pequenos.
            # Se for < 10kg, 1 pequeno resolve (mas cairia na entrega unica acima se tivesse estoque)

            dist_cd1 = instancia.distancias[('CD1', cliente)]
            dist_cd2 = instancia.distancias[('CD2', cliente)]
            cds_ordenados = sorted([('CD1', dist_cd1), ('CD2', dist_cd2)], key=lambda x: x[1])
            cd_pref, dist_pref = cds_ordenados[0]
            cd_seg, dist_seg = cds_ordenados[1]

            prods_cd1 = {}
            pedido_restante = pedido.copy()

            # Enche carrinho CD 1
            for prod, qtd in pedido.items():
                disponivel = solucao.estoque_atual[cd_pref][prod]
                if disponivel > 0:
                    qtd_pegar = min(qtd, disponivel)
                    prods_cd1[prod] = qtd_pegar
                    pedido_restante[prod] -= qtd_pegar
                    if pedido_restante[prod] == 0: del pedido_restante[prod]

            # Entrega Parte 1
            if prods_cd1:
                peso_parcial = sum(prods_cd1[p] * instancia.pesos[p] for p in prods_cd1)
                # Verifica se peso parcial cabe na moto (10kg)
                # Se não couber e não tiver médio -> Problema. Mas assumimos split equilibrado ou max 2 motos.
                # Simplificação: Se peso > 10 e sem médio, quebra de novo? O modelo limita a 2 splits.
                # Assumimos que o split resultará em cargas < 10kg.

                tipo_v = 'pequeno'
                if peso_parcial > 10:
                    # Se a parte 1 for > 10kg, precisamos de um médio.
                    if solucao.veiculos_utilizados['medio'] < limite_medios:
                        tipo_v = 'medio'
                    else:
                        # Crítico: Parte 1 > 10kg e sem caminhão.
                        # Não dá pra entregar essa parte. Abortar cliente.
                        continue

                solucao.adicionar_entrega(Entrega(cliente, cd_pref, tipo_v, prods_cd1, dist_pref))

            # Entrega Parte 2
            if pedido_restante:
                peso_restante = sum(pedido_restante[p] * instancia.pesos[p] for p in pedido_restante)

                tipo_v = 'pequeno'
                if peso_restante > 10:
                    if solucao.veiculos_utilizados['medio'] < limite_medios:
                        tipo_v = 'medio'
                    else:
                        # Se já entregou a parte 1, temos um problema de consistência (rollbak seria ideal).
                        # Como é heurística, aceitamos a falha ou usamos 'pequeno' com sobrepeso penalizado.
                        # Aqui vamos abortar para manter consistência rígida ou usar médio "fantasma" se preferir.
                        # Vamos usar 'medio' estourando limite apenas para não quebrar o código,
                        # ou continue (deixando pedido incompleto -> custo menor falso).
                        # Melhor: Tentar usar moto mesmo com peso excessivo e aplicar penalidade enorme no custo?
                        # Para simplificar: Tenta alocar Médio se sobrar, senão ignora (o que reduz custo incorretamente).
                        # Correção: Se chegou aqui, aloca médio e a contagem vai estourar (mostrando inviabilidade).
                        tipo_v = 'medio'

                # Verifica estoque CD 2
                tem_estoque_seg = True
                for prod, qtd in pedido_restante.items():
                    if solucao.estoque_atual[cd_seg][prod] < qtd: tem_estoque_seg = False

                if tem_estoque_seg:
                    solucao.adicionar_entrega(Entrega(cliente, cd_seg, tipo_v, pedido_restante, dist_seg))

    return solucao

# --- LOOP PRINCIPAL DA ALNS ---
def rodar_alns(instancia, modelo_tipo=1, max_iteracoes=1000):
    """
    modelo_tipo=1: Com Fracionamento (e teoricamente troca, mas nossa heurística faz direto)
    modelo_tipo=2: Sem Fracionamento (Entrega Única)
    modelo_tipo=3: Com Fracionamento, Sem Troca (Logicamente igual ao 1 na heurística)
    """
    # Modelo 1 permite troca. Modelos 2 e 3 não.
    permite_troca_cds = True if modelo_tipo == 1 else False

    solucao_atual = gerar_solucao_gulosa(instancia)
    melhor_solucao = copy.deepcopy(solucao_atual)

    # Modelo 1 e 3 permitem split. Modelo 2 não.
    permite_split = True if modelo_tipo in [1, 3] else False

    # Configuração de teste (para ser rápido)
    num_remover = 30

    print(f"  > Inicial: R$ {solucao_atual.custo_total:.2f} ({len(solucao_atual.entregas)} entregas)")

    for i in range(max_iteracoes):
        # Aviso de progresso a cada 10%
        if i > 0 and i % (max_iteracoes // 10) == 0:
            print(f"    ... processando {i}/{max_iteracoes} ...")

        # Cria Vizinho
        solucao_vizinha, clientes_removidos = destroy_random(solucao_atual, num_remover)

        # Reconstrói (com a regra do modelo!)
        solucao_vizinha = repair_greedy(solucao_vizinha, clientes_removidos, permitir_fracionamento=permite_split, permitir_troca=permite_troca_cds)

        # --- ADICIONE ESTA LINHA ---
        solucao_vizinha.recalcular_frota_do_zero()
        # Isso garante que a contagem seja exata para as entregas que estão na lista

        # Aceitação (Hill Climbing)
        if solucao_vizinha.custo_total < solucao_atual.custo_total:
            solucao_atual = solucao_vizinha
            # print(f"    Iter {i}: Melhorou! -> R$ {solucao_atual.custo_total:.2f}")

            if solucao_atual.custo_total < melhor_solucao.custo_total:
                melhor_solucao = copy.deepcopy(solucao_atual)

    return melhor_solucao

# --- BLOCO PRINCIPAL ---
if __name__ == "__main__":
    print("=== INICIANDO BATERIA COMPLETA (5 INSTÂNCIAS x 3 CENÁRIOS) ===")

    # Configurações
    ARQUIVO_PRODUTOS = "/content/olist_products_dataset.csv"
    MODELO_PARA_TESTAR = 1  # Mude para 1, 2 ou 3 conforme necessário
    ITERACOES = 100         # Use 100 para teste, 1000 para resultado final

    # Lista dos seus 5 arquivos de coordenadas
    arquivos_instancias = [
        "/content/Instance-Coordinates-0.csv",
        "/content/Instance-Coordinates-1.csv",
        "/content/Instance-Coordinates-2.csv",
        "/content/Instance-Coordinates-3.csv",
        "/content/Instance-Coordinates-4.csv"
    ]

    resultados_gerais = []

    # --- LOOP EXTERNO: INSTÂNCIAS ---
    for idx_arq, arquivo_end in enumerate(arquivos_instancias):
        nome_instancia = f"Instância {idx_arq + 1}"
        print(f"\n{'='*60}")
        print(f"PROCESSANDO: {nome_instancia} (Arquivo: {arquivo_end})")
        print(f"{'='*60}")

        # --- LOOP INTERNO: CENÁRIOS ---
        for cenario in [1, 2, 3]:
            print(f"\n  ---> Cenário {cenario} (Modelo {MODELO_PARA_TESTAR}) ...")

            instancia = Instancia(cenario_estoque=cenario, seed=42)
            try:
                # Carrega a instância atual do loop
                instancia.carregar_dados(arquivo_end, ARQUIVO_PRODUTOS)

                instancia.gerar_pedidos()
                instancia.gerar_estoque()
                instancia.calcular_frota()

                melhor_sol = rodar_alns(instancia, modelo_tipo=MODELO_PARA_TESTAR, max_iteracoes=ITERACOES)

                resultados_gerais.append({
                    "Instancia": nome_instancia,
                    "Cenario": cenario,
                    "Custo": melhor_sol.custo_total,
                    "Entregas": len(melhor_sol.entregas),
                    "Veiculos Pequenos": melhor_sol.veiculos_utilizados["pequeno"],
                    "Veiculos Médios": melhor_sol.veiculos_utilizados["medio"],
                    "Veiculos Grandes": melhor_sol.veiculos_utilizados["grande"]
                })
                print({
                          "Instancia": nome_instancia,
                          "Cenario": cenario,
                          "Custo": melhor_sol.custo_total,
                          "Entregas": len(melhor_sol.entregas),
                          "Veiculos Pequenos": melhor_sol.veiculos_utilizados["pequeno"],
                          "Veiculos Médios": melhor_sol.veiculos_utilizados["medio"],
                          "Veiculos Grandes": melhor_sol.veiculos_utilizados["grande"]
                      })


            except FileNotFoundError:
                print(f"ERRO: Arquivo {arquivo_end} não encontrado.")
                continue
            except Exception as e:
                print(f"Erro ao processar {nome_instancia}: {e}")
                continue

    # --- TABELA FINAL ---
    print("\n" + "="*70)
    print(f"RELATÓRIO GERAL - MODELO {MODELO_PARA_TESTAR}")
    print("="*70)
    print(f"{'Instância':<15} | {'Cenário':<8} | {'Custo (R$)':<12} | {'Entregas':<10}")
    print("-" * 70)

    for r in resultados_gerais:
        print(f"{r['Instancia']:<15} |    {r['Cenario']}     |  {r['Custo']:<12.2f} |    {r['Entregas']}")
    print("="*70)

=== INICIANDO BATERIA COMPLETA (5 INSTÂNCIAS x 3 CENÁRIOS) ===

PROCESSANDO: Instância 1 (Arquivo: /content/Instance-Coordinates-0.csv)

  ---> Cenário 1 (Modelo 1) ...
--- Carregando Dados Reais ---
Lendo produtos de: /content/olist_products_dataset.csv
Lendo endereços de: /content/Instance-Coordinates-0.csv
Sucesso! 100 produtos e 500 clientes carregados.
Calculando matriz de distâncias...
--- Gerando Pedidos (Tabela 1) ---
Pedidos gerados.
--- Gerando Estoque (Cenário 1) ---
Frota Calculada: Peq=1000, Med=4, Gra=4
--- Gerando Solução Inicial (Gulosa) ---
  > Inicial: R$ 4849.98 (662 entregas)
    ... processando 10/100 ...
    ... processando 20/100 ...
    ... processando 30/100 ...
    ... processando 40/100 ...
    ... processando 50/100 ...
    ... processando 60/100 ...
    ... processando 70/100 ...
    ... processando 80/100 ...
    ... processando 90/100 ...
{'Instancia': 'Instância 1', 'Cenario': 1, 'Custo': 3888.0139698677845, 'Entregas': 745, 'Veiculos Pequenos': 713, 'Ve